# 02 – Data Exploration

This notebook explores the raw customer support datasets used in the project.

The goal is to understand the structure, content, and noise characteristics of
the data before applying any preprocessing or information extraction techniques.


In [1]:
import pandas as pd
import re
import emoji

In [2]:
talkmap_path = "../data/raw/telecom_100k.csv"
comcast_path = "../data/raw/Comcast.csv"
bitext_path = "../data/raw/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv"

df_talkmap = pd.read_csv(talkmap_path)
df_comcast = pd.read_csv(comcast_path)
df_bitext = pd.read_csv(bitext_path)

In [3]:
emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags
                           u"\U00002700-\U000027BF"  # dingbats
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)

def extract_emojis(text):
    return emoji_pattern.findall(text)

In [40]:
patterns = {
    "urls": r"http[s]?://|www\.",          # any URL
    "repeated_punct": r"([!?])\1{1,}",     # !! or ??? or more
    "emails": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b",
    "numbers": r"\b\d+\b",
    "hashtags": r"#\w+",
    "mentions": r"@\w+",
    "emojis": "[" 
              u"\U0001F600-\U0001F64F"  # emoticons
              u"\U0001F300-\U0001F5FF"  # symbols & pictographs
              u"\U0001F680-\U0001F6FF"  # transport & map symbols
              u"\U0001F1E0-\U0001F1FF"  # flags
              u"\U00002700-\U000027BF"  # dingbats
              u"\U000024C2-\U0001F251"
              "]+"
}


In [41]:
def filter_by_pattern(df, column, pattern_name):
    regex = patterns[pattern_name]
    filtered = df[df[column].str.contains(regex, na=False)]
    return filtered

def show_examples(df, pattern_name, n=5):
    filtered = filter_by_pattern(df, TEXT_COL, pattern_name)
    print(f"--- {pattern_name} ({len(filtered)} messages) ---")
    display(filtered[[TEXT_COL]].head(n))


# talkmap Dataset

In [4]:
df_talkmap.columns


Index(['conversation_id', 'speaker', 'date_time', 'text'], dtype='object')

In [5]:
df_talkmap.head(10)


,conversation_id,speaker,date_time,text
0,cdb1999053df41aab7aa5ff569adef11,agent,2023-11-27T14:50:13.615384+00:00,"You're welcome, Mistie. I apologize again for ..."
1,b8a298e07cb24d02ada2c3a2bc857151,client,2023-10-25T13:09:34.692308+00:00,That sounds reassuring. But what if someone ha...
2,ff60f380ffb541e5a5f1a7f8a76ae77f,client,2023-09-11T12:24:37.538462+00:00,"Alright, thank you for your help, Dayna. I app..."
3,63a1604a78c4474389f2ba0d33919d41,agent,2023-09-04T15:01:57.153845+00:00,"Goodbye, Angeline. Have a great day."
4,793e6d7387fc4210994fba895e761c9d,agent,2023-11-18T08:06:38.692308+00:00,"You're welcome, Lessie. Thank you for choosing..."
5,760d0267ac97425f9183ad6456b3f53c,agent,2023-11-13T12:29:44.153845+00:00,I completely understand. Let me see if I can h...
6,d5c3c5d8fb9242848cf04e657bcd7824,client,2023-11-29T13:17:51.846153+00:00,"Thanks, you too."
7,cb48f3ab91ab4c4597961de562f9848c,agent,2023-11-05T09:52:27.538461+00:00,You're welcome! Thank you for choosing Union M...
8,f140ce86271f4e55a076e9f6ed8bba8d,agent,2023-10-12T16:40:26.538461+00:00,"Alright, Randell. Have a great day and good lu..."
9,12944af12a8a41fea558c7851a825e0d,client,2023-10-17T15:36:39.538462+00:00,Sure thing I'm actually going to Japan. I've h...


In [6]:
for i in range(10):
    print(f"\n--- Example {i+1} ---")
    print(df_talkmap.iloc[i]["text"])


--- Example 1 ---
You're welcome, Mistie. I apologize again for the inconvenience, and I appreciate your patience. I'll go ahead and transfer you now. (transfers call)

--- Example 2 ---
That sounds reassuring. But what if someone hacks into the app and gets access to my information?

--- Example 3 ---
Alright, thank you for your help, Dayna. I appreciate it.

--- Example 4 ---
Goodbye, Angeline. Have a great day.

--- Example 5 ---
You're welcome, Lessie. Thank you for choosing Union Mobile for your language translation needs. Have a great day!

--- Example 6 ---
I completely understand. Let me see if I can help you troubles the issue. Can you please verify your identity for me by providing your account PIN or the last four digits of the credit card associated with your account?

--- Example 7 ---
Thanks, you too.

--- Example 8 ---
You're welcome! Thank you for choosing Union Mobile, Tyron. Have a great day!

--- Example 9 ---
Alright, Randell. Have a great day and good luck with ge

In [7]:
print("\nNumber of rows with URLs in Talkmap dataset:")
df_talkmap[df_talkmap["text"].str.contains("http|www", na=False)].count()



Number of rows with URLs in Talkmap dataset:


conversation_id    6
speaker            6
date_time          6
text               6
dtype: int64

In [8]:
print("\nNumber of rows with emojis in Talkmap dataset:")
df_talkmap[df_talkmap["text"].str.contains("😂|😡|👍", na=False)].count()



Number of rows with emojis in Talkmap dataset:


conversation_id    5
speaker            5
date_time          5
text               5
dtype: int64

In [9]:
# add a column with emojis
df_talkmap["emojis"] = df_talkmap["text"].apply(lambda x: extract_emojis(str(x)))

In [10]:
# show messages containing at least one emoji
df_with_emojis = df_talkmap[df_talkmap["emojis"].map(len) > 0]

In [11]:
# count how many messages have emojis
print(df_with_emojis.shape[0])

24


In [12]:
# get a list of all unique emojis in the dataset
all_emojis = set([e for sublist in df_with_emojis["emojis"] for e in sublist])
print(all_emojis)

{'⟘�🌟😊�', '😘', '😍', '🌟😊👍🏼💯🎉👍🏼😄👍🏼💪🏽🎕🏾💪🏿💪🏼🌈🎞🌟😊👍🏼💕🎉👍🏼😊👍🏼💯🎯🌯🌟🌟🌟😊👍🏼💕🎉👍🏼😊👍🏼💪🏽💪🏾💪🏿💪🏼🌈🌞🌟😊👍🏼💕🎉👍🏼😊👍🏼💯💯💯🌟🌟🌟😊👍🏼💕🎉👍🏼😊👍🏼💪🏽💪🏾💪🏿💪🏼🌈🌞🌟😊👍🏼💕🎉👍🏼😊👍🏼💯💯💯🌟🌟🌟😊👍🏼💕🎉👍🏼😊👍🏼💪🏽💪🏾💪🏿💪🏼🌈🌞🌟😊👍🏼💕🎉👍🏼😊👍🏼💯💯💯🌟🌟🌟😊👍🏼💕🎉👍🏼😊👍🏼💪🏽💪🏾💪🏿💪🏼🌈🌞🌟😊👍🏼💕🎉👍🏼😊👍🏼💯💯💯🌟🌟🌟😊👍🏼💕🎉👍🏼😊👍🏼💪🏽💪🏾💪🏿💪🏼🌈🌞🌟😊👍🏼💕🎉👍🏼😊👍🏼💯💯💯🌟🌟🌟😊👍🏼💕🎉👍🏼😊👍🏼💪🏽💪🏾💪🏿💪🏼🌈🌞🌟😊👍🏼💕🎉👍🏼😊👍🏼💯💯💯🌟🌟🌟😊👍🏼💕🎉👍🏼😊👍🏼💪🏽💪🏾💪🏿💪🏼🌈🌞🌟😊👍🏼💕🎉👍🏼😊👍🏼💯💯💯🌟🌟🌟😊👍🏼💕🎉👍🏼😊👍🏼💪🏽💪🏾💪🏿💪🏼🌈🌞🌟😊👍🏼💕🎉👍🏼😊👍🏼💯💯💯🌟🌟🌟😊👍🏼💕🎉👍🏼😊👍🏼💪🏽💪🏾💪🏿💪🏼🌈🌞🌟😊👍🏼💕🎉👍🏼😊👍🏼💯💯💯🌟🌟🌟😊👍🏼💕🎉👍🏼😊👍🏼💪🏽💪🏾💪🏿💪🏼🌈🌞🌟😊👍🏼💕🎉👍🏼😊👍🏼💯💯💯🌟🌟🌟😊👍🏼💕🎉👍🏼😊👍🏼💪🏽💪🏾💪🏿💪🏼🌈🌞🌟😊👍🏼💕🎉👍🏼😊👍🏼💯💯💯🌟🌟🌟😊👍🏼💕🎉👍🏼😊👍🏼💪🏽💪🏾💪🏿💪🏼🌈🌞🌟😊👍🏼💕🎉👍🏼😊👍🏼💯💯💯🌟🌟🌟😊👍🏼💕🎉👍🏼😊👍🏼💪🏽💪🏾💪🏿💪🏼🌈🌞🌟😊👍🏼💕🎉👍🏼😊👍🏼💯💯💯🌟🌟🌟😊👍🏼💕🎉👍🏼😊👍🏼💪🏽💪🏾💪🏿💪🏼🌈🌞🌟���', '�', '😊', '💯', '📞', '😊📞💻👬', '👍💻📞🔋👀🎉💯', '😄', '👍💻📞🔋👀🎉💯💬👏🌟😊', '���', '👏', '👋', '🙌', '💕', '👍💻📞🔋', '😊👍💻📱🎧🔋✨💯🙊👍🏼💬👀', '💋', '🚀', '💬', '���💻📞🔋👀🎉💯💬👏🌟😊', '🌟', '🎉', '📞👍💻🎉😬', '👍'}


In [42]:
TEXT_COL = "text"

In [43]:
for name in patterns:
    show_examples(df_talkmap, name, n=5)

--- urls (5 messages) ---


,text
722,"Sure, here's the error message: ""Invalid reque..."
6214,"Alright, here's the first one: ""http://www.exa..."
19096,"Sure thing, Maranda. Our website is [www.union..."
23693,"Now, enter the following information exactly a..."
58301,Of course! here's the URL to Lookout's website...


--- repeated_punct (65 messages) ---


C:\Users\nourg\AppData\Local\Temp\ipykernel_13392\2619430470.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  filtered = df[df[column].str.contains(regex, na=False)]


,text
420,"Hi, I' wanted to follow up on the call forward..."
856,"Sorry to hear that, Elna. Can you please provi..."
1784,Let me check. ( about tomorrow at 2 PM??
4588,That sounds good. What other plans do you have??
6153,"Hi Kelsey, thank you for reaching out to Union..."


...

,text
241,"Sure, my account number is #1234567890."
381,"Sure, it's #1234567890."
521,"Sure. my account number is #1234567890, and my..."
755,"Sure, my account number is #1234567890. I'm tr..."
820,"Sure, my account number is #1234567890. And I'..."


--- mentions (97 messages) ---


,text
147,"Yes, my email address is [ingrid@email.com](ma..."
252,(impuctantly) Fine. It's [sammie@email.com](ma...
1347,"Sure, it's [florance@xyzenterprises.com](mailt..."
1920,(sighs) Fine. My email address is [charles@ema...
5028,"Yes, my contact address is [georgianacustomer@..."


--- emojis (24 messages) ---


,text
1792,"(to herself) Whew, that was a bit one! I'm gla..."
5158,You're welcome! Duncan! Enjoy your gaming expe...
6256,"😊 Thanks for letting me know, Clifford. I'm gl..."
6829,You're welcome! Thank you for choosing Union M...
10459,"(to herself) Well, that was a bit of a challen..."


In [44]:
counts = {name: len(filter_by_pattern(df_talkmap, TEXT_COL, name)) for name in patterns}
pd.DataFrame(list(counts.items()), columns=["Pattern", "Message Count"])

C:\Users\nourg\AppData\Local\Temp\ipykernel_13392\2619430470.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  filtered = df[df[column].str.contains(regex, na=False)]


,Pattern,Message Count
0,urls,5
1,repeated_punct,65
2,emails,92
3,numbers,8702
4,hashtags,434
5,mentions,97
6,emojis,24


# Comcast Dataset

In [13]:
df_comcast.columns

Index(['Ticket #', 'Customer Complaint', 'Date', 'Date_month_year', 'Time',
       'Received Via', 'City', 'State', 'Zip code', 'Status',
       'Filing on Behalf of Someone'],
      dtype='object')

In [14]:
df_comcast.head(10)

,Ticket #,Customer Complaint,Date,Date_month_year,Time,Received Via,City,State,Zip code,Status,Filing on Behalf of Someone
0,250635,Comcast Cable Internet Speeds,22-04-15,22-Apr-15,3:53:50 PM,Customer Care Call,Abingdon,Maryland,21009,Closed,No
1,223441,Payment disappear - service got disconnected,04-08-15,04-Aug-15,10:22:56 AM,Internet,Acworth,Georgia,30102,Closed,No
2,242732,Speed and Service,18-04-15,18-Apr-15,9:55:47 AM,Internet,Acworth,Georgia,30101,Closed,Yes
3,277946,Comcast Imposed a New Usage Cap of 300GB that ...,05-07-15,05-Jul-15,11:59:35 AM,Internet,Acworth,Georgia,30101,Open,Yes
4,307175,Comcast not working and no service to boot,26-05-15,26-May-15,1:25:26 PM,Internet,Acworth,Georgia,30101,Solved,No
5,338519,ISP Charging for arbitrary data limits with ov...,06-12-15,06-Dec-15,9:59:40 PM,Internet,Acworth,Georgia,30101,Solved,No
6,361148,Throttling service and unreasonable data caps,24-06-15,24-Jun-15,10:13:55 AM,Customer Care Call,Acworth,Georgia,30101,Pending,No
7,359792,Comcast refuses to help troubleshoot and corre...,23-06-15,23-Jun-15,6:56:14 PM,Internet,Adrian,Michigan,49221,Solved,No
8,318072,Comcast extended outages,06-01-15,06-Jan-15,11:46:30 PM,Customer Care Call,Alameda,California,94502,Closed,No
9,371214,Comcast Raising Prices and Not Being Available...,28-06-15,28-Jun-15,6:46:31 PM,Customer Care Call,Alameda,California,94501,Open,Yes


In [16]:
for i in range(10):
    print(f"\n--- Example {i+1} ---")
    print(df_comcast.iloc[i]["Customer Complaint"])


--- Example 1 ---
Comcast Cable Internet Speeds

--- Example 2 ---
Payment disappear - service got disconnected

--- Example 3 ---
Speed and Service

--- Example 4 ---
Comcast Imposed a New Usage Cap of 300GB that punishes streaming.

--- Example 5 ---
Comcast not working and no service to boot

--- Example 6 ---
ISP Charging for arbitrary data limits with overage fees

--- Example 7 ---
Throttling service and unreasonable data caps

--- Example 8 ---
Comcast refuses to help troubleshoot and correct my service.

--- Example 9 ---
Comcast extended outages

--- Example 10 ---
Comcast Raising Prices and Not Being Available To Ask Why


In [17]:
print("\nNumber of rows with URLs in Comcast dataset:")
df_comcast[df_comcast["Customer Complaint"].str.contains("http|www", na=False)].count()


Number of rows with URLs in Comcast dataset:


Ticket #                       0
Customer Complaint             0
Date                           0
Date_month_year                0
Time                           0
Received Via                   0
City                           0
State                          0
Zip code                       0
Status                         0
Filing on Behalf of Someone    0
dtype: int64

In [19]:
print("\nNumber of rows with emojis in Comcast dataset:")
df_comcast[df_comcast["Customer Complaint"].str.contains("😂|😡|👍", na=False)].count()


Number of rows with emojis in Comcast dataset:


Ticket #                       0
Customer Complaint             0
Date                           0
Date_month_year                0
Time                           0
Received Via                   0
City                           0
State                          0
Zip code                       0
Status                         0
Filing on Behalf of Someone    0
dtype: int64

In [20]:
# add a column with emojis
df_comcast["emojis"] = df_comcast["Customer Complaint"].apply(lambda x: extract_emojis(str(x)))

In [21]:
# show messages containing at least one emoji
df_comcast_with_emojis = df_comcast[df_comcast["emojis"].map(len) > 0]

In [22]:
# count how many messages have emojis
print(df_comcast_with_emojis.shape[0])

0


In [23]:
# get a list of all unique emojis in the dataset
all_emojis_comcast = set([e for sublist in df_comcast_with_emojis["emojis"] for e in sublist])
print(all_emojis_comcast)

set()


In [45]:
TEXT_COL = "Customer Complaint"

In [46]:
for name in patterns:
    show_examples(df_comcast, name, n=5)


--- urls (0 messages) ---


,Customer Complaint


--- repeated_punct (4 messages) ---


C:\Users\nourg\AppData\Local\Temp\ipykernel_13392\2619430470.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  filtered = df[df[column].str.contains(regex, na=False)]


,Customer Complaint
223,Comcast keeps changing bill and every time giv...
996,LIED TO!!! Now I'm suffering?!?! And at a loss!!!
1429,The WORST customer service team and negative o...
1517,Comcast keeps hiking my bill for no reason !!


...

,Customer Complaint
1482,Comcast terrible service #2
1776,Postal Mail Ticket - #0057
1786,Postal Mail Ticket #2334
1988,Complaint#67455
2139,Complaint #2. No resolution. Please help. Than...


--- mentions (1 messages) ---


,Customer Complaint
270,COMCAST BILLING FOR LAST @18 mos for services ...


--- emojis (0 messages) ---


,Customer Complaint


In [47]:
counts = {name: len(filter_by_pattern(df_comcast, TEXT_COL, name)) for name in patterns}
pd.DataFrame(list(counts.items()), columns=["Pattern", "Message Count"])


C:\Users\nourg\AppData\Local\Temp\ipykernel_13392\2619430470.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  filtered = df[df[column].str.contains(regex, na=False)]


,Pattern,Message Count
0,urls,0
1,repeated_punct,4
2,emails,0
3,numbers,89
4,hashtags,5
5,mentions,1
6,emojis,0


# Bitext Dataset

In [24]:
df_bitext.columns

Index(['flags', 'instruction', 'category', 'intent', 'response'], dtype='object')

In [25]:
df_bitext.head(10)

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...
5,BI,can you help me cancel order {{Order Number}}?,ORDER,cancel_order,"Of course, I'm here to assist you in canceling..."
6,BCLN,"I can no longer afford order {{Order Number}},...",ORDER,cancel_order,I pick up what you're putting down that you're...
7,BL,I am trying to cancel purchase {{Order Number}},ORDER,cancel_order,I've understood that you're seeking assistance...
8,BL,I have got to cancel purchase {{Order Number}},ORDER,cancel_order,I'm sensitive to the fact that you're seeking ...
9,BLQ,i need help canceling purchase {{Order Number}},ORDER,cancel_order,I perceive that you're seeking assistance with...


In [26]:
for i in range(10):
    print(f"\n--- Example {i+1} ---")
    print(df_bitext.iloc[i]["instruction"])


--- Example 1 ---
question about cancelling order {{Order Number}}

--- Example 2 ---
i have a question about cancelling oorder {{Order Number}}

--- Example 3 ---
i need help cancelling puchase {{Order Number}}

--- Example 4 ---
I need to cancel purchase {{Order Number}}

--- Example 5 ---
I cannot afford this order, cancel purchase {{Order Number}}

--- Example 6 ---
can you help me cancel order {{Order Number}}?

--- Example 7 ---
I can no longer afford order {{Order Number}}, cancel it

--- Example 8 ---
I am trying to cancel purchase {{Order Number}}

--- Example 9 ---
I have got to cancel purchase {{Order Number}}

--- Example 10 ---
i need help canceling purchase {{Order Number}}


In [27]:
print("\nNumber of rows with URLs in Bitext dataset:")
df_bitext[df_bitext["instruction"].str.contains("http|www", na=False)].count()


Number of rows with URLs in Bitext dataset:


flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64

In [28]:
print("\nNumber of rows with emojis in Bitext dataset:")
df_bitext[df_bitext["instruction"].str.contains("😂|😡|👍", na=False)].count()


Number of rows with emojis in Bitext dataset:


flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64

In [29]:
# add a column with emojis
df_bitext["emojis"] = df_bitext["instruction"].apply(lambda x: extract_emojis(str(x)))

In [30]:
# show messages containing at least one emoji
df_bitext_with_emojis = df_bitext[df_bitext["emojis"].map(len) > 0]

In [31]:
# count how many messages have emojis
print(df_bitext_with_emojis.shape[0])

0


In [32]:
# get a list of all unique emojis in the dataset
all_emojis_bitext = set([e for sublist in df_bitext_with_emojis["emojis"] for e in sublist])
print(all_emojis_bitext)

set()


In [48]:
TEXT_COL = "instruction"

In [49]:
for name in patterns:
    show_examples(df_bitext, name, n=5)


--- urls (0 messages) ---


,instruction


--- repeated_punct (0 messages) ---


C:\Users\nourg\AppData\Local\Temp\ipykernel_13392\2619430470.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  filtered = df[df[column].str.contains(regex, na=False)]


,instruction


...

,instruction
3922,how can I locate the invoice #37777?
3924,i need assistance giving a quick look at the b...
3925,can you help me check bill #12588?
3926,locate the bill #12588
3927,want assistance giving a quick look at the bil...


--- mentions (0 messages) ---


,instruction


--- emojis (0 messages) ---


,instruction


In [50]:
counts = {name: len(filter_by_pattern(df_bitext, TEXT_COL, name)) for name in patterns}
pd.DataFrame(list(counts.items()), columns=["Pattern", "Message Count"])


C:\Users\nourg\AppData\Local\Temp\ipykernel_13392\2619430470.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  filtered = df[df[column].str.contains(regex, na=False)]


,Pattern,Message Count
0,urls,0
1,repeated_punct,0
2,emails,0
3,numbers,1045
4,hashtags,1037
5,mentions,0
6,emojis,0


In [33]:
for i in range(10):
    print(f"\n--- Example {i+1} ---")
    print(df_bitext.iloc[i]["response"])


--- Example 1 ---
I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.

--- Example 2 ---
I've been informed that you have a question about canceling order {{Order Number}}. I'm here to assist you! Please go ahead and let me know what specific question you have, and I'll provide you with all the information and guidance you need. Your satisfaction is my top priority.

--- Example 3 ---
I can sense that you're seeking assistance with canceling your purchase with the purchase number {{Order Number}}. I apologize for any inconvenience caused, and I'm here to guide you through the process.

To cancel your purchase, please follow these steps:

1. Log into your account: Visit our {{Online Company Portal Info}} and sign in using your credentials.
2. Locate your order: Once logged in, navigate to the '{{Online Order Interaction}}' or '

In [34]:
print("\nNumber of rows with URLs in Bitext dataset:")
df_bitext[df_bitext["response"].str.contains("http|www", na=False)].count()


Number of rows with URLs in Bitext dataset:


flags          5
instruction    5
category       5
intent         5
response       5
emojis         5
dtype: int64

In [35]:
print("\nNumber of rows with emojis in Bitext dataset:")
df_bitext[df_bitext["response"].str.contains("😂|😡|👍", na=False)].count()


Number of rows with emojis in Bitext dataset:


flags          1
instruction    1
category       1
intent         1
response       1
emojis         1
dtype: int64

In [55]:
# add a column with emojis
df_bitext["emojis"] = df_bitext["response"].apply(lambda x: extract_emojis(str(x)))

In [56]:
# show messages containing at least one emoji
df_bitext_with_emojis = df_bitext[df_bitext["emojis"].map(len) > 0]

In [57]:
# count how many messages have emojis
print(df_bitext_with_emojis.shape[0])

62


In [58]:
# get a list of all unique emojis in the dataset
all_emojis_bitext = set([e for sublist in df_bitext_with_emojis["emojis"] for e in sublist])
print(all_emojis_bitext)

{'😊', '🗝️', '🔐💡', '☺️', '🙏', '🛡️🔒', '🌟', '💪🔐', '🙁', '💪', '👍', '✨'}


In [51]:
TEXT_COL = "response"

In [52]:
for name in patterns:
    show_examples(df_bitext, name, n=5)


--- urls (5 messages) ---


,response
11665,We've received your message to us regarding th...
12861,I'll make it happen! I'm here to guide you on ...
20903,We understand that you're experiencing some di...
21067,I'm glad you contacted us to us regarding the ...
25077,Thank you for reaching out! I'm glad you're ta...


--- repeated_punct (0 messages) ---


C:\Users\nourg\AppData\Local\Temp\ipykernel_13392\2619430470.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  filtered = df[df[column].str.contains(regex, na=False)]


,response


...

,response


--- mentions (55 messages) ---


,response
8013,Thank you for reaching out! I completely under...
8195,We're here for you! I'm aware that you have a ...
9892,Your message means a lot! I'm aligned with the...
11665,We've received your message to us regarding th...
18092,"To inform us about a payment issue, you can re..."


--- emojis (62 messages) ---


,response
8945,Thank you for getting in touch! 🌟 I understand...
9068,Thank you for getting in touch! 🌟 I understand...
9076,I'm sorry to hear that you feel that way. 🙁 Ou...
9208,Thank you for your inquiry! 🌟 I understand tha...
9209,Thank you for contacting! I certainly recogniz...


In [53]:
counts = {name: len(filter_by_pattern(df_bitext, TEXT_COL, name)) for name in patterns}
pd.DataFrame(list(counts.items()), columns=["Pattern", "Message Count"])


C:\Users\nourg\AppData\Local\Temp\ipykernel_13392\2619430470.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  filtered = df[df[column].str.contains(regex, na=False)]


,Pattern,Message Count
0,urls,5
1,repeated_punct,0
2,emails,55
3,numbers,6367
4,hashtags,0
5,mentions,55
6,emojis,62
